# Optogenetic stimulation only for programs that drive the LED

The published conversion adds an `OptogeneticSeries` at the reward times to every opto session except the probe programs. Only 14 MED-PC programs used in the opto experiments switch on the LED output (`ON ^STIM` in `MSNs/*.MPC`); the FR1 and RI programs without STIM or SCRAM in their names do not, so those sessions received no light. Separately, scrambled sessions whose stimulation array `Z` is empty got a one-sample, zero-power series.

This notebook finds both kinds of published series, reconverts those sessions with the fixed code, and reconverts a few stimulation sessions to check that their series is unchanged.

In [1]:
from pathlib import Path
import random
import shutil

import h5py
import numpy as np
import pandas as pd

from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_dataset import (
    opto_to_nwb,
    get_no_port_entry_duration_sessions,
    get_session_key_from_kwargs,
)
from lerner_lab_to_nwb.seiler_2024.seiler_2024_convert_session import session_to_nwb

data_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/_Raw_Data_for_future_reference")
published_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/000971")
output_dir_path = Path("/Volumes/T7/CatalystNeuro/Lerner/fix_audit/opto_programs")
if output_dir_path.exists():
    shutil.rmtree(output_dir_path)
output_dir_path.mkdir(parents=True)
stimulation_msns = {
    "FR1_BOTH_SCRAMBLED",
    "FR1_BOTH_WStim",
    "FR1_LEFT_SCRAM",
    "FR1_LEFT_STIM",
    "FR1_RIGHT_SCRAMBLED",
    "FR1_RIGHT_STIM",
    "RI 30 LEFT_STIM",
    "RI 30 RIGHT_STIM",
    "RI 60 LEFT_STIM",
    "RI 60 RIGHT STIM",
    "RI30 Left Scrambled",
    "RI30 Right SCRAMBLED",
    "RI60_LEFT_SCRAM",
    "RI60_RIGHT_SCRAM",
}

In [2]:
published_rows = []
for file_path in sorted(published_dir_path.glob("sub-*/*_ses-Opto-*.nwb")):
    with h5py.File(file_path, "r") as file:
        if "stimulus/presentation/OptogeneticSeries" not in file:
            continue
        msn = file["processing/behavior"].attrs["description"].split("MSN = ")[1].split("\n")[0].strip()
        data = file["stimulus/presentation/OptogeneticSeries/data"][()]
    published_rows.append(
        dict(file_path=file_path, msn=msn, samples=len(data), zero_power=not np.any(data))
    )
published_df = pd.DataFrame(published_rows)
published_df["stimulation_program"] = published_df["msn"].isin(stimulation_msns)
published_df.groupby(["stimulation_program", "zero_power"]).size()

stimulation_program  zero_power
False                False          166
True                 False         2235
                     True             8
dtype: int64

In [3]:
published_df[~published_df["stimulation_program"]].groupby("msn").size()

msn
FOOD_FR1 HT TTL (Both)    26
FOOD_FR1 TTL Left         84
FOOD_FR1 TTL Right        53
FOOD_RI 60 LEFT TTL        1
FOOD_RI 60 RIGHT TTL       2
dtype: int64

In [4]:
published_df[published_df["zero_power"]].assign(file=lambda df: df["file_path"].map(lambda path: path.name))[["file", "msn", "samples"]]

,file,msn,samples
526,sub-230-431_ses-Opto-DMS-Inhibitory-NpHRScramb...,FR1_LEFT_SCRAM,1
529,sub-230-431_ses-Opto-DMS-Inhibitory-NpHRScramb...,FR1_RIGHT_SCRAMBLED,1
601,sub-239-388_ses-Opto-DLS-Excitatory-ChR2Scramb...,FR1_LEFT_SCRAM,1
1141,sub-294-406_ses-Opto-DMS-Excitatory-ChR2Scramb...,RI60_RIGHT_SCRAM,1
1377,sub-322-415_ses-Opto-DMS-Excitatory-ChR2Scramb...,FR1_RIGHT_SCRAMBLED,1
1379,sub-322-415_ses-Opto-DMS-Excitatory-ChR2Scramb...,FR1_RIGHT_SCRAMBLED,1
1444,sub-341-401_ses-Opto-DLS-Excitatory-ChR2Scramb...,FR1_RIGHT_SCRAMBLED,1
1973,sub-431-419_ses-Opto-DMS-Excitatory-ChR2Scramb...,FR1_RIGHT_SCRAMBLED,1


## Reconvert those sessions

In [5]:
session_to_nwb_kwargs_per_session = opto_to_nwb(
    data_dir_path=data_dir_path, output_dir_path=output_dir_path, start_variable="Start Date", verbose=False
)
start_to_kwargs = {}
for session_to_nwb_kwargs in session_to_nwb_kwargs_per_session:
    session_conditions = session_to_nwb_kwargs["session_conditions"]
    start = pd.Timestamp(f"{session_conditions['Start Date']} {session_conditions['Start Time']}").isoformat()
    key = (session_to_nwb_kwargs["subject_id"], start.replace(":", "-"), session_to_nwb_kwargs["optogenetic_treatment"])
    start_to_kwargs[key] = session_to_nwb_kwargs


def find_kwargs(published_file_path: Path) -> dict:
    subject_part, session_part = published_file_path.stem.split("_ses-")
    subject_id = subject_part.removeprefix("sub-").replace("-", ".")
    session_id = session_part.split("_")[0]
    return start_to_kwargs[(subject_id, session_id[-19:], session_id.split("-")[3])]


random.seed(0)
affected_df = published_df[~published_df["stimulation_program"] | published_df["zero_power"]]
unaffected_df = published_df.loc[random.sample(list(published_df[published_df["stimulation_program"] & ~published_df["zero_power"]].index), 5)]
selected_df = pd.concat([affected_df, unaffected_df])
selected_kwargs = [find_kwargs(file_path) for file_path in selected_df["file_path"]]
no_port_entry_duration_sessions = get_no_port_entry_duration_sessions(
    port_entry_file_path=output_dir_path / "sessions_without_port_entry_durations.yaml",
    session_to_nwb_args_per_session=selected_kwargs,
)
for session_to_nwb_kwargs in selected_kwargs:
    if get_session_key_from_kwargs(session_to_nwb_kwargs) in no_port_entry_duration_sessions:
        session_to_nwb_kwargs["has_port_entry_durations"] = False
    session_to_nwb(**session_to_nwb_kwargs)

Reading port entry durations:   0%|          | 0/179 [00:00<?, ?it/s]

Reading port entry durations:   8%|▊         | 14/179 [00:00<00:01, 129.59it/s]

Reading port entry durations:  15%|█▌        | 27/179 [00:00<00:01, 120.02it/s]

Reading port entry durations:  23%|██▎       | 42/179 [00:00<00:01, 131.79it/s]

Reading port entry durations:  31%|███▏      | 56/179 [00:00<00:01, 119.52it/s]

...

Reading port entry durations:  69%|██████▊   | 123/179 [00:01<00:00, 124.90it/s]

Reading port entry durations:  77%|███████▋  | 138/179 [00:01<00:00, 132.02it/s]

Reading port entry durations:  85%|████████▍ | 152/179 [00:01<00:00, 104.19it/s]

Reading port entry durations:  94%|█████████▍| 169/179 [00:01<00:00, 117.00it/s]

Reading port entry durations: 100%|██████████| 179/179 [00:01<00:00, 114.50it/s]

In [6]:
rows = []
for published_file_path, published_row in zip(selected_df["file_path"], selected_df.itertuples()):
    subject_part, session_part = published_file_path.stem.split("_ses-")
    subject_id = subject_part.removeprefix("sub-").replace("-", ".")
    session_id = session_part.split("_")[0]
    (new_file_path,) = output_dir_path.glob(f"sub-{subject_id}_ses-{session_id}.nwb")
    with h5py.File(published_file_path, "r") as published_file, h5py.File(new_file_path, "r") as new_file:
        new_has_series = "stimulus/presentation/OptogeneticSeries" in new_file
        series_equal = new_has_series and all(
            np.array_equal(
                published_file[f"stimulus/presentation/OptogeneticSeries/{name}"][()],
                new_file[f"stimulus/presentation/OptogeneticSeries/{name}"][()],
            )
            for name in ("data", "timestamps")
        )
        rows.append(
            dict(
                category=(
                    "zero power"
                    if published_row.zero_power
                    else "stimulation program" if published_row.stimulation_program else "non-stimulation program"
                ),
                new_has_series=new_has_series,
                new_has_site="general/optogenetics/OptogeneticStimulusSite" in new_file,
                series_equal_to_published=series_equal,
            )
        )
pd.DataFrame(rows).value_counts().to_frame("sessions")

,,,,sessions
category,new_has_series,new_has_site,series_equal_to_published,
non-stimulation program,False,True,False,166
zero power,False,False,False,8
stimulation program,True,True,True,5
